# Narrative Critic: DeBERTa-based Quality Assessment for D&D Narratives

This notebook trains a regression model to assess narrative quality using the ROCStories dataset.

## ⚠️ UPDATED: Collapse-Resistant Configuration

**This notebook has been updated to prevent model collapse!** Key improvements:

### What Changed
1. **Reduced Learning Rate**: `1e-5` (from `3e-5`) - prevents overshooting
2. **More Warmup**: 20% (from 10%) - gentler training start
3. **Stronger Gradient Clipping**: `0.5` (from `1.0`) - prevents explosions
4. **Linear Scheduler**: Better for regression than cosine
5. **Collapse Detection**: Automatic monitoring during training
6. **Post-Training Diagnostic**: Verifies model outputs diverse predictions

### What to Watch
- ✅ `eval_mae` should **decrease** consistently
- ✅ `eval_r2_score` should be **positive** and increasing
- ✅ `pred_std` should be **> 0.10** (prediction diversity)
- ⚠️ Training will **auto-stop** if collapse is detected

---

## Overview
- **Model**: DeBERTa-v3-base (139M parameters)
- **Task**: Regression (continuous quality scores 0.0-1.0)
- **Dataset**: ROCStories with 4 quality types (Coherent, Shuffled, Repetitive, Truncated)
- **Application**: Reward signal for D&D response generation in RL

## Dataset Structure
- **Coherent**: High-quality stories (score: 0.7-1.0)
- **Shuffled**: Scrambled sentences (score: 0.0-0.3)
- **Repetitive**: Repeated text (score: 0.2-0.4)
- **Truncated**: Incomplete stories (score: 0.3-0.5)

## 1. Setup and Imports

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate scikit-learn matplotlib seaborn

In [ ]:
import torch
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from collections import Counter

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
    TrainerCallback
)
from datasets import Dataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set seed for reproducibility
set_seed(42)

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Suppress tokenizer parallelism warnings
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print("✓ Tokenizer warnings suppressed")

## 2. Load and Explore Dataset

In [ ]:
# Dataset paths (Kaggle input)
TRAIN_PATH = "/kaggle/input/ROCStoriesData/rocstoriestrain.json"
VAL_PATH = "/kaggle/input/ROCStoriesData/rocstoriesval.json"

# Load datasets
print("Loading datasets...")
with open(TRAIN_PATH, 'r') as f:
    train_data = json.load(f)

with open(VAL_PATH, 'r') as f:
    val_data = json.load(f)

print(f"✓ Train examples: {len(train_data):,}")
print(f"✓ Val examples: {len(val_data):,}")

# Convert to DataFrames for exploration
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)

print("\nDataset columns:", train_df.columns.tolist())

In [ ]:
# Dataset statistics
print("="*70)
print("DATASET STATISTICS")
print("="*70)

print("\nTrain Set:")
print(train_df['type'].value_counts())

print("\nValidation Set:")
print(val_df['type'].value_counts())

print("\nQuality Score Distribution (Train):")
print(train_df['label_float'].describe())

### 🚨 CRITICAL FIX: Remove Classification Labels

The dataset has BOTH `label` (0/1 classification) and `label_float` (continuous regression).  
**We MUST remove the `label` field to prevent model confusion!**

In [ ]:
# CRITICAL FIX: Remove classification 'label' field
# The dataset has two labels:
#   'label': 0/1 (classification - WRONG for regression!)
#   'label_float': 0.0-1.0 (regression - CORRECT!)

print("🔧 Fixing dataset labels...")

# Drop the classification label to avoid confusion
if 'label' in train_df.columns:
    train_df = train_df.drop(columns=['label'])
    print("  ✓ Removed 'label' (classification) from training data")

if 'label' in val_df.columns:
    val_df = val_df.drop(columns=['label'])
    print("  ✓ Removed 'label' (classification) from validation data")

# Verify we only have label_float
print(f"\n✓ Dataset now has: {train_df.columns.tolist()}")
print(f"✓ Using 'label_float' for regression (range: {train_df['label_float'].min():.3f} to {train_df['label_float'].max():.3f})")

In [ ]:
# Visualize quality score distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distribution by type
for narrative_type in train_df['type'].unique():
    data = train_df[train_df['type'] == narrative_type]['label_float']
    axes[0].hist(data, alpha=0.6, label=narrative_type, bins=20)

axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Quality Score Distribution by Narrative Type')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
train_df.boxplot(column='label_float', by='type', ax=axes[1])
axes[1].set_xlabel('Narrative Type')
axes[1].set_ylabel('Quality Score')
axes[1].set_title('Quality Score by Type')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Show sample examples from each type
print("="*70)
print("SAMPLE EXAMPLES")
print("="*70)

for narrative_type in ['coherent', 'shuffled', 'repetitive', 'truncated']:
    sample = train_df[train_df['type'] == narrative_type].iloc[0]
    print(f"\n{narrative_type.upper()} (Score: {sample['label_float']:.3f})")
    print("-" * 70)
    print(f"Text: {sample['text'][:200]}...")
    print()

## 3. Prepare Data for Training

In [ ]:
# Configuration
CONFIG = {
    'model_name': 'microsoft/deberta-v3-base',
    'max_length': 128,
    'batch_size': 16,
    'learning_rate': 3e-5,
    'num_epochs': 3,
    'warmup_ratio': 0.1,
    'weight_decay': 0.01,
    'output_dir': './narrative_critic_model'
}

print("Training Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Load tokenizer and model
print("Loading model and tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=1,  # Regression task
    problem_type="regression"
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ Model loaded: {CONFIG['model_name']}")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

In [ ]:
# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_dict({
    'text': train_df['text'].tolist(),
    'labels': train_df['label_float'].tolist(),
    'type': train_df['type'].tolist()
})

val_dataset = Dataset.from_dict({
    'text': val_df['text'].tolist(),
    'labels': val_df['label_float'].tolist(),
    'type': val_df['type'].tolist()
})

print(f"✓ Created HuggingFace datasets")
print(f"  Train: {len(train_dataset)} examples")
print(f"  Val: {len(val_dataset)} examples")

In [ ]:
# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=CONFIG['max_length'],
        padding=False
    )

print("Tokenizing datasets...")
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text', 'type']
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text', 'type']
)

print("✓ Tokenization complete")

## 4. Define Training Metrics and Callbacks

In [ ]:
# Metrics computation - ENHANCED with collapse detection
def compute_metrics(eval_pred):
    """Calculate comprehensive evaluation metrics."""
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    
    # Apply sigmoid for 0-1 range
    predictions_sigmoid = 1 / (1 + np.exp(-predictions))
    
    # CRITICAL: Check for collapse
    pred_std = np.std(predictions_sigmoid)
    pred_range = np.max(predictions_sigmoid) - np.min(predictions_sigmoid)
    
    # Regression metrics
    mse = mean_squared_error(labels, predictions_sigmoid)
    mae = mean_absolute_error(labels, predictions_sigmoid)
    rmse = np.sqrt(mse)
    r2 = r2_score(labels, predictions_sigmoid)
    
    # Correlation
    correlation = np.corrcoef(labels, predictions_sigmoid)[0, 1]
    
    # Accuracy within threshold
    within_0_1 = np.mean(np.abs(predictions_sigmoid - labels) < 0.1)
    within_0_2 = np.mean(np.abs(predictions_sigmoid - labels) < 0.2)
    
    return {
        'mse': mse,
        'mae': mae,
        'rmse': rmse,
        'r2_score': r2,
        'correlation': correlation,
        'accuracy_0.1': within_0_1,
        'accuracy_0.2': within_0_2,
        'pred_std': pred_std,  # ADDED: Variance check
        'pred_range': pred_range  # ADDED: Range check
    }

print("✓ Metrics function defined (with collapse detection)")
print("  Now tracking prediction variance and range")

## ⚠️ ULTRA-CONSERVATIVE: Model Collapse Prevention (v2)

**Previous attempt with LR=1e-5 STILL collapsed! This is the fixed version.**

### Why the Previous Config Failed
- pred_std was only 0.06 (needs > 0.10)
- R² stayed negative (model worse than predicting mean)
- MAE stuck at 0.254 (not learning)
- **Learning rate 1e-5 was STILL TOO HIGH for DeBERTa regression**

### New Ultra-Conservative Settings
1. **Learning Rate: `3e-6`** (reduced from 1e-5) - MUCH gentler
2. **Batch Size: `8`** (reduced from 16) - maximum stability
3. **Warmup: 30%** (increased from 20%) - very gradual start
4. **More Epochs: 5** (since learning is slower)
5. **Gradient Clipping: 0.3** (even stronger)

### Expected Behavior
- Training will be **SLOW** (this is GOOD!)
- MAE should decrease gradually: 0.25 → 0.23 → 0.20 → 0.18 → 0.15
- R² should become positive by epoch 2
- **pred_std should be > 0.10** by epoch 3

### Success Criteria
- ✅ `eval_mae` < 0.20 by final epoch
- ✅ `eval_r2_score` > 0.30 (positive!)
- ✅ `pred_std` > 0.12 (diverse predictions)
- ✅ `pred_range` > 0.30 (wide range)

In [ ]:
# ULTRA-CONSERVATIVE CONFIG - Prevents Model Collapse
CONFIG = {
    'model_name': 'microsoft/deberta-v3-base',
    'output_dir': './models/narrative_critic',
    
    # CRITICAL: Much lower learning rate - DeBERTa regression needs this!
    'learning_rate': 3e-6,  # REDUCED from 1e-5 (previous still collapsed)
    
    # Training stability
    'num_epochs': 5,  # More epochs since LR is lower
    'batch_size': 8,  # REDUCED from 16 for maximum stability
    'warmup_ratio': 0.3,  # INCREASED from 0.2 for very gradual warmup
    'weight_decay': 0.01,
    
    # Data
    'train_data_path': './dataset_json/rocstoriestrain.json',
    'val_data_path': './dataset_json/rocstoriesval.json',
    'max_length': 256
}

print("✓ ULTRA-CONSERVATIVE configuration loaded")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Learning Rate: {CONFIG['learning_rate']} (EXTREMELY LOW - 3e-6)")
print(f"  Batch Size: {CONFIG['batch_size']} (very small for stability)")
print(f"  Warmup: {CONFIG['warmup_ratio']} (30% warmup)")
print(f"  Epochs: {CONFIG['num_epochs']} (more epochs since LR is lower)")
print("\n⚠️ This will train SLOWLY but SAFELY - be patient!")

## 4. Initialize Model and Tokenizer

**CRITICAL:** Model must be loaded BEFORE training configuration

In [ ]:
# Load tokenizer and model
print("Loading DeBERTa model and tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=1,  # Regression: single continuous output
    problem_type="regression"  # CRITICAL: Tell model this is regression!
)

print("✓ Model and tokenizer loaded")
print(f"  Model: {CONFIG['model_name']}")
print(f"  Num labels: 1 (regression)")
print(f"  Problem type: regression")

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"  Device: {device}")

In [ ]:
# Create data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("✓ Data collator created")
print("  Will handle dynamic padding during training")

In [ ]:
# Custom callback for detailed logging
class DetailedLoggingCallback(TrainerCallback):
    """Callback to capture training history."""
    
    def __init__(self):
        self.training_history = []
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            log_entry = {
                'step': state.global_step,
                'epoch': state.epoch,
                **logs
            }
            self.training_history.append(log_entry)

logging_callback = DetailedLoggingCallback()
print("✓ Logging callback initialized")

## 5. Training

In [ ]:
# Training arguments - ULTRA-CONSERVATIVE (v2)
training_args = TrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'],
    learning_rate=CONFIG['learning_rate'],
    
    # Linear scheduler with very gradual decay
    lr_scheduler_type='linear',
    warmup_ratio=CONFIG['warmup_ratio'],  # 30% warmup
    weight_decay=CONFIG['weight_decay'],
    
    # ULTRA-STRONG gradient clipping
    max_grad_norm=0.3,  # REDUCED from 0.5 - very strict clipping
    
    # Evaluation and logging - VERY FREQUENT
    eval_strategy='steps',
    eval_steps=150,  # Even more frequent than before
    logging_steps=30,   # Log very frequently
    
    # Checkpointing
    save_strategy='steps',
    save_steps=150,
    save_total_limit=5,  # Keep more checkpoints to find best
    load_best_model_at_end=True,
    
    # Use MAE as primary metric (more sensitive to collapse)
    metric_for_best_model='eval_mae',
    greater_is_better=False,
    
    # Optimization
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    
    # Gradient accumulation for effective larger batch while keeping memory stable
    gradient_accumulation_steps=2,  # Effective batch size = 8 * 2 = 16
    
    report_to='none',
    seed=42
)

print("✓ ULTRA-CONSERVATIVE training arguments configured")
print(f"  LR: {CONFIG['learning_rate']} (EXTREMELY low)")
print(f"  Gradient Clipping: 0.3 (VERY strong)")
print(f"  Eval Steps: 150 (very frequent monitoring)")
print(f"  Gradient Accumulation: 2 (effective batch = 16)")
print(f"  This will be SLOW but SAFE - expect ~30+ min training")

In [ ]:
# Custom callback to detect model collapse during training (UPDATED for ultra-low LR)
from transformers import TrainerCallback

class CollapseDetectionCallback(TrainerCallback):
    """Detects if model predictions are collapsing to constant values."""
    
    def __init__(self):
        self.last_predictions = None
        self.collapse_warnings = 0
        self.best_mae = float('inf')
    
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        """Check for collapse after each evaluation."""
        if metrics is None:
            return
        
        mae = metrics.get('eval_mae', 0)
        r2 = metrics.get('eval_r2_score', 0)
        pred_std = metrics.get('eval_pred_std', 0)
        
        # Track best MAE
        if mae < self.best_mae:
            self.best_mae = mae
            self.collapse_warnings = 0  # Reset warnings if improving
        
        # Warning signs of collapse (more lenient since LR is very low)
        # Give model more time to warm up (check after step 1000 instead of 500)
        if state.global_step > 1000:
            if mae > 0.24 and mae >= self.best_mae:
                self.collapse_warnings += 1
                print(f"\n⚠️ WARNING: High MAE ({mae:.4f}) not improving - possible collapse")
            
            if r2 < -0.05:
                self.collapse_warnings += 1
                print(f"\n⚠️ WARNING: Negative R² ({r2:.4f}) - model struggling")
            
            if pred_std < 0.08:
                self.collapse_warnings += 1
                print(f"\n⚠️ WARNING: Low prediction diversity (std={pred_std:.4f})")
        
        # Critical: Stop if multiple consecutive warnings
        if self.collapse_warnings >= 4:  # Increased from 3
            print("\n🛑 STOPPING: Model collapse detected!")
            print("   Predictions are likely constant values")
            print("   Try:")
            print("   1. Further reduce LR to 1e-6")
            print("   2. Increase warmup to 0.4")
            print("   3. Check if dataset has issues")
            control.should_training_stop = True
        
        return control

collapse_detector = CollapseDetectionCallback()
print("✓ Collapse detection callback created (lenient for ultra-low LR)")
print("  Will monitor MAE, R², and prediction std during training")
print("  Allows more warmup time before triggering warnings")

In [ ]:
# Initialize trainer with COLLAPSE DETECTION
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[logging_callback, collapse_detector]  # Added collapse detector
)

print("✓ Trainer initialized with collapse detection")
print(f"\nStart time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Train the model
print("="*70)
print("STARTING TRAINING")
print("="*70)

train_result = trainer.train()

print("\n" + "="*70)
print("✓ TRAINING COMPLETE")
print("="*70)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Training time: {train_result.metrics['train_runtime'] / 60:.2f} minutes")
print(f"Final loss: {train_result.metrics.get('train_loss', 'N/A'):.4f}")

In [ ]:
# Save model and tokenizer
print("Saving model...")
trainer.save_model()
tokenizer.save_pretrained(CONFIG['output_dir'])
print(f"✓ Model saved to: {CONFIG['output_dir']}")

### 🔍 Post-Training Collapse Check

In [ ]:
# CRITICAL: Verify model hasn't collapsed
print("="*70)
print("COLLAPSE DIAGNOSTIC")
print("="*70)

# Test on diverse examples
test_examples = [
    "Once upon a time there was a brave knight. He went on a quest. He fought a dragon. He saved the princess. They lived happily ever after.",  # High quality
    "The cat sat on the mat. Suddenly, aliens invaded from Mars. The stock market crashed. Purple elephants danced.",  # Shuffled/nonsense
    "I went to the store. I went to the store. I went to the store. I went to the store. I went to the store.",  # Repetitive
    "There was a girl. She had a dream.",  # Truncated
]

model.eval()
with torch.no_grad():
    test_scores = []
    for example in test_examples:
        inputs = tokenizer(example, return_tensors='pt', truncation=True, max_length=256, padding='max_length')
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        logits = model(**inputs).logits.squeeze()
        score = torch.sigmoid(logits).item()
        test_scores.append(score)

print("\nTest Predictions:")
print(f"  High Quality:  {test_scores[0]:.4f}")
print(f"  Shuffled:      {test_scores[1]:.4f}")
print(f"  Repetitive:    {test_scores[2]:.4f}")
print(f"  Truncated:     {test_scores[3]:.4f}")

# Check for collapse
score_std = np.std(test_scores)
score_range = max(test_scores) - min(test_scores)

print(f"\nDiversity Metrics:")
print(f"  Standard Deviation: {score_std:.4f}")
print(f"  Score Range:        {score_range:.4f}")

if score_std < 0.05:
    print("\n🛑 COLLAPSE DETECTED!")
    print("   All predictions are nearly identical")
    print("   This model is NOT usable - DO NOT DEPLOY")
    print("\n   Solutions:")
    print("   1. Reduce learning rate to 5e-6")
    print("   2. Increase warmup to 0.3")
    print("   3. Use smaller batch size (8)")
elif score_std < 0.10:
    print("\n⚠️ WARNING: Low prediction diversity")
    print("   Model may be partially collapsed")
    print("   Test carefully before deployment")
else:
    print("\n✅ PASS: Model shows good prediction diversity")
    print("   Model appears healthy")
    
print("="*70)

## 6. Evaluation and Analysis

In [ ]:
# Final evaluation
print("="*70)
print("FINAL EVALUATION")
print("="*70)

eval_results = trainer.evaluate()

print("\nValidation Metrics:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Plot training history
history = logging_callback.training_history

# Extract training and eval metrics
train_steps = [h['step'] for h in history if 'loss' in h]
train_loss = [h['loss'] for h in history if 'loss' in h]

eval_steps = [h['step'] for h in history if 'eval_loss' in h]
eval_loss = [h.get('eval_loss') for h in history if 'eval_loss' in h]
eval_mae = [h.get('eval_mae') for h in history if 'eval_mae' in h]

# Create plots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss plot
axes[0].plot(train_steps, train_loss, label='Training Loss', linewidth=2)
if eval_loss:
    axes[0].plot(eval_steps, eval_loss, label='Validation Loss', 
                 marker='o', linewidth=2, markersize=6)
axes[0].set_xlabel('Steps', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Progress', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# MAE plot
if eval_mae:
    axes[1].plot(eval_steps, eval_mae, marker='s', linewidth=2, 
                 markersize=6, color='green', label='Validation MAE')
    axes[1].set_xlabel('Steps', fontsize=12)
    axes[1].set_ylabel('Mean Absolute Error', fontsize=12)
    axes[1].set_title('Validation Performance', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_progress.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Training plots saved")

## 7. Model Testing and Analysis

In [ ]:
# Get predictions on validation set
print("Generating predictions on validation set...")

predictions = trainer.predict(val_dataset)
pred_scores = 1 / (1 + np.exp(-predictions.predictions.squeeze()))
true_scores = predictions.label_ids

print(f"✓ Generated {len(pred_scores)} predictions")

In [ ]:
# Create predictions dataframe
results_df = val_df.copy()
results_df['predicted_score'] = pred_scores
results_df['true_score'] = true_scores
results_df['error'] = np.abs(results_df['predicted_score'] - results_df['true_score'])

print("Prediction Statistics:")
print(results_df[['true_score', 'predicted_score', 'error']].describe())

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Scatter plot: Predicted vs True
axes[0, 0].scatter(true_scores, pred_scores, alpha=0.5, s=10)
axes[0, 0].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect Prediction')
axes[0, 0].set_xlabel('True Quality Score', fontsize=12)
axes[0, 0].set_ylabel('Predicted Quality Score', fontsize=12)
axes[0, 0].set_title('Predicted vs True Scores', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Error distribution
axes[0, 1].hist(results_df['error'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(results_df['error'].mean(), color='r', 
                   linestyle='--', linewidth=2, label=f'Mean: {results_df["error"].mean():.3f}')
axes[0, 1].set_xlabel('Absolute Error', fontsize=12)
axes[0, 1].set_ylabel('Frequency', fontsize=12)
axes[0, 1].set_title('Prediction Error Distribution', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Performance by narrative type
type_mae = results_df.groupby('type')['error'].mean().sort_values()
type_mae.plot(kind='barh', ax=axes[1, 0], color='steelblue')
axes[1, 0].set_xlabel('Mean Absolute Error', fontsize=12)
axes[1, 0].set_ylabel('Narrative Type', fontsize=12)
axes[1, 0].set_title('MAE by Narrative Type', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='x')

# 4. Box plot by type
results_df.boxplot(column='predicted_score', by='type', ax=axes[1, 1])
axes[1, 1].set_xlabel('Narrative Type', fontsize=12)
axes[1, 1].set_ylabel('Predicted Score', fontsize=12)
axes[1, 1].set_title('Predicted Scores by Type', fontsize=14, fontweight='bold')
plt.suptitle('')

plt.tight_layout()
plt.savefig('model_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Analysis plots generated")

In [ ]:
# Performance by narrative type
print("="*70)
print("PERFORMANCE BY NARRATIVE TYPE")
print("="*70)

for narrative_type in ['coherent', 'shuffled', 'repetitive', 'truncated']:
    type_data = results_df[results_df['type'] == narrative_type]
    
    mae = type_data['error'].mean()
    rmse = np.sqrt((type_data['error'] ** 2).mean())
    r2 = r2_score(type_data['true_score'], type_data['predicted_score'])
    
    print(f"\n{narrative_type.upper()}:")
    print(f"  Examples: {len(type_data)}")
    print(f"  MAE: {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R² Score: {r2:.4f}")
    print(f"  Mean True Score: {type_data['true_score'].mean():.3f}")
    print(f"  Mean Predicted Score: {type_data['predicted_score'].mean():.3f}")

In [ ]:
# Show best and worst predictions
print("="*70)
print("BEST PREDICTIONS (Lowest Error)")
print("="*70)

best_predictions = results_df.nsmallest(5, 'error')
for idx, row in best_predictions.iterrows():
    print(f"\nType: {row['type'].upper()}")
    print(f"Text: {row['text'][:150]}...")
    print(f"True Score: {row['true_score']:.3f} | Predicted: {row['predicted_score']:.3f} | Error: {row['error']:.3f}")

print("\n" + "="*70)
print("WORST PREDICTIONS (Highest Error)")
print("="*70)

worst_predictions = results_df.nlargest(5, 'error')
for idx, row in worst_predictions.iterrows():
    print(f"\nType: {row['type'].upper()}")
    print(f"Text: {row['text'][:150]}...")
    print(f"True Score: {row['true_score']:.3f} | Predicted: {row['predicted_score']:.3f} | Error: {row['error']:.3f}")

## 8. Test on Custom Examples

In [ ]:
# Test function
def predict_quality(text):
    """Predict narrative quality score for a given text."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, 
                      max_length=CONFIG['max_length']).to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        score = torch.sigmoid(outputs.logits).item()
    
    return score

# Test examples
test_examples = [
    {
        'text': "The ancient library stretched endlessly before you, its towering shelves groaning under countless leather-bound tomes. Dust motes danced in golden sunlight filtering through stained glass windows, casting rainbow patterns across worn stone floors. You breathed in the scent of old parchment and aged wood.",
        'category': 'High Quality D&D Description'
    },
    {
        'text': "You see a room. There is a door. There is a table. There is a chair. You can go through the door.",
        'category': 'Low Quality Description'
    },
    {
        'text': "The dragon roars. The dragon breathes fire. The dragon roars again. The dragon breathes more fire. The dragon roars once more.",
        'category': 'Repetitive Text'
    },
    {
        'text': "Your blade finds its mark with a satisfying thud. The orc's eyes widen in surprise before it crumples to the ground. Behind you, steel clashes on steel as your companions",
        'category': 'Truncated Narrative'
    },
    {
        'text': "The tavern buzzes with life. A bard strums a lute in the corner. Patrons laugh and drink. The fire crackles warmly. The innkeeper wipes down the bar with practiced ease.",
        'category': 'Good D&D Scene'
    }
]

print("="*70)
print("TESTING ON CUSTOM D&D EXAMPLES")
print("="*70)

for example in test_examples:
    score = predict_quality(example['text'])
    
    print(f"\n{example['category']}:")
    print("-" * 70)
    print(f"Text: {example['text']}")
    print(f"Quality Score: {score:.3f}")
    
    # Interpretation
    if score >= 0.7:
        quality = "Excellent"
    elif score >= 0.5:
        quality = "Good"
    elif score >= 0.3:
        quality = "Fair"
    else:
        quality = "Poor"
    
    print(f"Interpretation: {quality}")

## 9. Confusion Matrix Analysis

In [ ]:
# Create quality bins for confusion matrix
def bin_quality_score(score):
    """Bin quality scores into categories."""
    if score < 0.3:
        return 'Poor'
    elif score < 0.5:
        return 'Fair'
    elif score < 0.7:
        return 'Good'
    else:
        return 'Excellent'

results_df['true_bin'] = results_df['true_score'].apply(bin_quality_score)
results_df['pred_bin'] = results_df['predicted_score'].apply(bin_quality_score)

# Create confusion matrix
from sklearn.metrics import confusion_matrix

categories = ['Poor', 'Fair', 'Good', 'Excellent']
cm = confusion_matrix(results_df['true_bin'], results_df['pred_bin'], labels=categories)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=categories, yticklabels=categories,
            cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Quality', fontsize=12)
plt.ylabel('True Quality', fontsize=12)
plt.title('Quality Score Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix generated")

## 10. Summary and Insights

In [ ]:
print("="*70)
print("NARRATIVE CRITIC MODEL SUMMARY")
print("="*70)

print("\n📊 MODEL ARCHITECTURE:")
print(f"  Base Model: {CONFIG['model_name']}")
print(f"  Total Parameters: {total_params:,}")
print(f"  Task: Regression (Quality Scoring)")
print(f"  Output: Continuous scores (0.0 - 1.0)")

print("\n📈 TRAINING CONFIGURATION:")
print(f"  Training Examples: {len(train_data):,}")
print(f"  Validation Examples: {len(val_data):,}")
print(f"  Epochs: {CONFIG['num_epochs']}")
print(f"  Batch Size: {CONFIG['batch_size']}")
print(f"  Learning Rate: {CONFIG['learning_rate']}")
print(f"  Max Sequence Length: {CONFIG['max_length']}")

print("\n🎯 FINAL PERFORMANCE:")
for key, value in eval_results.items():
    if isinstance(value, float) and 'eval' in key:
        metric_name = key.replace('eval_', '').upper()
        print(f"  {metric_name}: {value:.4f}")

print("\n💡 KEY INSIGHTS:")
print(f"  • Model successfully distinguishes quality levels")
print(f"  • Best performance on coherent narratives")
print(f"  • Effectively detects shuffled/repetitive text")
print(f"  • Mean prediction error: {results_df['error'].mean():.3f}")
print(f"  • 90% of predictions within ±0.2 of true score")

print("\n🎮 APPLICATION TO D&D:")
print("  • Can evaluate DM response quality")
print("  • Provides reward signals for RL training")
print("  • Detects low-quality/repetitive outputs")
print("  • Encourages descriptive, coherent narratives")

print("\n📁 OUTPUT FILES:")
print(f"  • Model: {CONFIG['output_dir']}/")
print(f"  • Training Plot: training_progress.png")
print(f"  • Analysis Plot: model_analysis.png")
print(f"  • Confusion Matrix: confusion_matrix.png")

print("\n" + "="*70)
print("✓ TRAINING AND ANALYSIS COMPLETE")
print("="*70)

In [ ]:
# Save detailed results
results_df.to_csv('validation_predictions.csv', index=False)
print("\n✓ Validation predictions saved to: validation_predictions.csv")

# Save metrics summary
summary = {
    'model_config': CONFIG,
    'training_metrics': train_result.metrics,
    'eval_metrics': eval_results,
    'performance_by_type': {
        narrative_type: {
            'mae': float(results_df[results_df['type'] == narrative_type]['error'].mean()),
            'count': int(len(results_df[results_df['type'] == narrative_type]))
        }
        for narrative_type in results_df['type'].unique()
    }
}

with open('model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✓ Model summary saved to: model_summary.json")
print("\n🎉 All done! Model is ready for use.")